# OpenAI Responses API: Maintaining Conversation with `previous_response_id`

This notebook shows how to build **multi-turn conversations** with the OpenAI **Responses API** using `previous_response_id`.

You will learn how to:

- send a first request with `client.responses.create(...)`
- continue the conversation by passing `previous_response_id=response.id`
- understand how `instructions` behave across turns
- create a reusable helper for stateful chat
- inspect response IDs and usage
- compare **server-side conversation state** vs manually resending history

> This notebook is designed to be practical and runnable, with examples you can adapt into scripts, apps, or assignments.


## Notebook Outline

1. Environment setup  
2. Why `previous_response_id` matters  
3. Minimal single-turn example  
4. Multi-turn conversation example  
5. Important behavior of `instructions`  
6. Building a reusable conversation helper  
7. Inspecting responses and metadata  
8. Optional manual-history comparison  
9. Best practices and common pitfalls


In [3]:
import os
from dotenv import load_dotenv

# Load variables from the .env file into the system environment
load_dotenv() 

# Retrieve the key using standard os.getenv
api_key = os.getenv('OPENAI_API_KEY')

# 3. Verify exactly like you did in Colab
if api_key:
    print(f"API Key loaded successfully! Starts with: {api_key[:7]}...")
else:
    print("API Key not found. Make sure your .env file is in the same folder.")

API Key loaded successfully! Starts with: sk-proj...


In [4]:
# If needed, install/upgrade the OpenAI Python SDK:
# %pip install -U openai
import os
from pprint import pprint
from typing import Optional, List, Dict, Any
from dotenv import load_dotenv, find_dotenv

from openai import OpenAI

_ = load_dotenv(find_dotenv())

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY is not set. Set it before running live API calls.")
else:
    print("OpenAI client initialized.")

OpenAI client initialized.


## Why use `previous_response_id`?

When you create a response, OpenAI returns a **response object** with an **ID**.  
You can pass that ID into the next call as `previous_response_id` to continue the conversation without manually resending the full prior exchange every time.

This is useful for:

- chat-style applications
- tutoring or Q&A sessions
- stateful demos in notebooks
- cleaner code when you do not want to rebuild message history manually


In [5]:
MODEL = "gpt-4.1-mini"  # You can change this to another supported model.

response_1 = client.responses.create(
    model=MODEL,
    input="In one short paragraph, explain what an embedding is in machine learning."
)

print("Response ID:", response_1.id)
print()
print(response_1.output_text)


Response ID: resp_0e18d989861f44020069dcd46a36b8819099fa017b429cff85

In machine learning, an embedding is a dense, low-dimensional vector representation of high-dimensional or discrete data, such as words, images, or graphs. Embeddings capture the underlying semantic or structural relationships by mapping similar items to nearby points in the vector space, enabling algorithms to process and analyze complex data more effectively. For example, word embeddings represent words as vectors where those with similar meanings have similar representations, facilitating tasks like natural language understanding and recommendation systems.


## Multi-turn example using `previous_response_id`

The next call refers to the first response by ID.  
That lets the model continue the same conversation context.


In [6]:
response_2 = client.responses.create(
    model=MODEL,
    previous_response_id=response_1.id,
    input="Now explain the same idea as if you were teaching a beginner who knows basic Excel."
)

print("Previous response ID used:", response_1.id)
print("New response ID:", response_2.id)
print()
print(response_2.output_text)


Previous response ID used: resp_0e18d989861f44020069dcd46a36b8819099fa017b429cff85
New response ID: resp_0e18d989861f44020069dcd47421108190b9f36c3e3fb1ea59

Think of an embedding like turning words or other information into a list of numbers that shows how they relate to each other, kind of like how you might use colors or shapes in Excel to group similar things. Instead of using long complicated descriptions, the embedding gives you a simple row of numbers that computers can easily work with. For example, if you had a list of fruits in Excel, an embedding would turn each fruit into numbers so that similar fruits like “apple” and “pear” have numbers that are close to each other, helping the computer understand their connection.


In [7]:
response_3 = client.responses.create(
    model=MODEL,
    previous_response_id=response_2.id,
    input="Give me 3 real-world examples of where embeddings are useful."
)

print("Chained from:", response_2.id)
print("Current response ID:", response_3.id)
print()
print(response_3.output_text)


Chained from: resp_0e18d989861f44020069dcd47421108190b9f36c3e3fb1ea59
Current response ID: resp_0e18d989861f44020069dcd47722d48190af16b0ad577f5531

Here are three real-world examples where embeddings are useful:

1. **Search Engines**: Embeddings help improve search results by understanding the meaning behind your queries and matching them to relevant documents, even if the exact words don’t appear.

2. **Recommendation Systems**: Platforms like Netflix or Spotify use embeddings to represent users and content, so they can recommend movies, songs, or products that are similar to your preferences.

3. **Language Translation**: Embeddings capture the meaning of words and sentences in different languages, enabling more accurate and natural translations by machines.


## Important behavior: `instructions` are **not automatically carried forward**

This is one of the most important details when using `previous_response_id`.

If you set `instructions` on one call and then create a new turn with `previous_response_id`, the old instructions are **not automatically inherited**.  
So if you need a stable role or style, pass `instructions` again on each turn where you want them enforced.


In [8]:
# First turn with explicit instructions
response_a = client.responses.create(
    model=MODEL,
    instructions="You are a patient math tutor. Always answer using bullet points and simple language.",
    input="Explain overfitting."
)

print("Turn A ID:", response_a.id)
print(response_a.output_text)


Turn A ID: resp_0cad62b1924940360069dcd47b83548194be1807ce243a00b0
Sure! Here's a simple explanation of overfitting:

- Overfitting happens when a model learns the training data too well.
- It captures not only the real patterns but also the noise or random details.
- Because of this, the model works great on the training data but poorly on new data.
- Think of it like memorizing answers instead of understanding the topic.
- Overfitting makes the model less flexible and less useful in real situations.
- To avoid overfitting, we can use methods like:
  - Using simpler models
  - Getting more training data
  - Using techniques like regularization or cross-validation

Let me know if you want me to explain any of these ideas more!


In [9]:
# Follow-up WITHOUT repeating instructions
response_b = client.responses.create(
    model=MODEL,
    previous_response_id=response_a.id,
    input="Now explain underfitting."
)

print("Turn B ID:", response_b.id)
print(response_b.output_text)

print("\nNOTE: Depending on model behavior, style may drift here because instructions were not repeated.")


Turn B ID: resp_0cad62b1924940360069dcd4803cc481948f3a998013601740
Certainly! Here's an explanation of **underfitting**:

- Underfitting happens when a model is too simple to capture the underlying patterns in the data.
- It means the model cannot learn enough from the training data.
- As a result, the model performs poorly both on the training data and on new, unseen data.
- It’s like having an oversimplified explanation that misses important details.
- Underfitting indicates that the model has high bias and low complexity.
- To fix underfitting, you can:
  - Use a more complex model
  - Add more relevant features
  - Train longer or reduce regularization if it’s too strong

In short, underfitting means the model is not powerful enough to represent the data well. Let me know if you want examples or more details!

NOTE: Depending on model behavior, style may drift here because instructions were not repeated.


In [10]:
# Follow-up WITH repeated instructions
response_c = client.responses.create(
    model=MODEL,
    previous_response_id=response_b.id,
    instructions="You are a patient math tutor. Always answer using bullet points and simple language.",
    input="Now compare overfitting and underfitting."
)

print("Turn C ID:", response_c.id)
print(response_c.output_text)


Turn C ID: resp_0cad62b1924940360069dcd483645081949d1885fef499148f
Sure! Here’s a simple comparison of **overfitting** and **underfitting**:

- **Overfitting:**
  - Model learns too much from the training data, including noise.
  - Works very well on training data but poorly on new data.
  - Model is too complex.
  - High variance, low bias.
  - Example: Memorizing answers instead of understanding.

- **Underfitting:**
  - Model learns too little and misses important patterns.
  - Performs poorly on both training and new data.
  - Model is too simple.
  - High bias, low variance.
  - Example: Giving an oversimplified explanation that misses details.

- **Goal in machine learning:** Find the balance between overfitting and underfitting for best generalization to new data.

Let me know if you want a visual or more explanation!


## Reusable helper class for stateful conversation

The helper below stores the latest response ID and reuses it automatically.
This is a convenient pattern for notebooks, prototypes, and small apps.


In [11]:
class StatefulResponsesChat:
    def __init__(self, client: OpenAI, model: str, instructions: Optional[str] = None):
        self.client = client
        self.model = model
        self.instructions = instructions
        self.last_response_id: Optional[str] = None
        self.history: List[Dict[str, Any]] = []

    def ask(self, user_input: str, instructions: Optional[str] = None, **kwargs):
        active_instructions = instructions if instructions is not None else self.instructions

        request_kwargs = {
            "model": self.model,
            "input": user_input,
            **kwargs,
        }

        if self.last_response_id:
            request_kwargs["previous_response_id"] = self.last_response_id

        # Repeat instructions when you want them enforced on this turn.
        if active_instructions:
            request_kwargs["instructions"] = active_instructions

        response = self.client.responses.create(**request_kwargs)

        self.history.append(
            {
                "user_input": user_input,
                "response_id": response.id,
                "output_text": response.output_text,
            }
        )

        self.last_response_id = response.id
        return response

    def reset(self):
        self.last_response_id = None
        self.history = []

    def show_history(self):
        for i, item in enumerate(self.history, start=1):
            print(f"Turn {i}")
            print("User:", item["user_input"])
            print("Response ID:", item["response_id"])
            print("Assistant:", item["output_text"])
            print("-" * 80)


In [12]:
chat = StatefulResponsesChat(
    client=client,
    model=MODEL,
    instructions="You are a concise teaching assistant for an advanced deep learning class."
)

r1 = chat.ask("What is the difference between supervised and unsupervised learning?")
print(r1.output_text)

print("\n" + "="*80 + "\n")

r2 = chat.ask("Give me one example of each in healthcare.")
print(r2.output_text)

print("\n" + "="*80 + "\n")

r3 = chat.ask("Now summarize both in a 2-row table.")
print(r3.output_text)

print("\nLatest response ID:", chat.last_response_id)


**Supervised Learning**: The model is trained on labeled data, meaning each input has a corresponding target output. The goal is to learn a mapping from inputs to outputs (e.g., classification, regression).

**Unsupervised Learning**: The model is trained on unlabeled data without explicit targets. It aims to find hidden patterns or structures in the data (e.g., clustering, dimensionality reduction).


**Supervised Learning example:**  
Predicting patient disease diagnosis from medical images (e.g., classifying X-rays as healthy vs. pneumonia) using labeled image data.

**Unsupervised Learning example:**  
Identifying patient subgroups with similar symptoms or genetic profiles via clustering to discover new disease phenotypes without predefined labels.


| Learning Type       | Healthcare Example                                     |
|---------------------|-------------------------------------------------------|
| Supervised Learning  | Diagnosing diseases from labeled medical images  

In [13]:
chat.show_history()


Turn 1
User: What is the difference between supervised and unsupervised learning?
Response ID: resp_0bc4e3252ca588270069dcd48db238819098368659232fb4ac
Assistant: **Supervised Learning**: The model is trained on labeled data, meaning each input has a corresponding target output. The goal is to learn a mapping from inputs to outputs (e.g., classification, regression).

**Unsupervised Learning**: The model is trained on unlabeled data without explicit targets. It aims to find hidden patterns or structures in the data (e.g., clustering, dimensionality reduction).
--------------------------------------------------------------------------------
Turn 2
User: Give me one example of each in healthcare.
Response ID: resp_0bc4e3252ca588270069dcd49038788190bb57d3224f5dc107
Assistant: **Supervised Learning example:**  
Predicting patient disease diagnosis from medical images (e.g., classifying X-rays as healthy vs. pneumonia) using labeled image data.

**Unsupervised Learning example:**  
Identifyi

## Inspecting response details

You may want to inspect IDs, usage, status, and output structure for debugging or logging.


In [14]:
example_response = r3

print("ID:", example_response.id)
print("Status:", getattr(example_response, "status", None))
print("Model:", getattr(example_response, "model", None))
print("Previous Response ID:", getattr(example_response, "previous_response_id", None))

print("\nUsage:")
pprint(getattr(example_response, "usage", None))

print("\nTop-level keys (if available via model_dump):")
if hasattr(example_response, "model_dump"):
    pprint(list(example_response.model_dump().keys()))
else:
    print("model_dump() not available in this SDK version.")


ID: resp_0bc4e3252ca588270069dcd4923cc88190bf2d04bdfb0428e6
Status: completed
Model: gpt-4.1-mini-2025-04-14
Previous Response ID: resp_0bc4e3252ca588270069dcd49038788190bb57d3224f5dc107

Usage:
ResponseUsage(input_tokens=218, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=48, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=266)

Top-level keys (if available via model_dump):
['id',
 'created_at',
 'error',
 'incomplete_details',
 'instructions',
 'metadata',
 'model',
 'object',
 'output',
 'parallel_tool_calls',
 'temperature',
 'tool_choice',
 'tools',
 'top_p',
 'background',
 'completed_at',
 'conversation',
 'max_output_tokens',
 'max_tool_calls',
 'previous_response_id',
 'prompt',
 'prompt_cache_key',
 'prompt_cache_retention',
 'reasoning',
 'safety_identifier',
 'service_tier',
 'status',
 'text',
 'top_logprobs',
 'truncation',
 'usage',
 'user',
 'billing',
 'frequency_penalty',
 'presence_penalty',
 'store']


## Optional comparison: manual history vs `previous_response_id`

You can also keep the full conversation yourself and resend it manually as structured `input`.
That gives you more direct control, but it is more verbose.

In many notebook and prototype scenarios, `previous_response_id` is simpler.


In [15]:
manual_history = [
    {"role": "user", "content": "Teach me gradient descent in simple terms."},
    {"role": "assistant", "content": "Gradient descent is a method for gradually improving model parameters by reducing error step by step."},
    {"role": "user", "content": "Give me a real-world analogy."}
]

response_manual = client.responses.create(
    model=MODEL,
    instructions="You are a friendly tutor. Keep answers brief.",
    input=manual_history
)

print(response_manual.output_text)


Sure! Imagine you’re on a foggy mountain and want to reach the lowest point (valley). You can’t see far, so you feel the slope under your feet and take small steps downhill. Repeating this helps you get closer to the valley. That’s like gradient descent finding the minimum error by moving step-by-step in the best direction.


## Best practices

- **Save the latest response ID** if you plan to continue a conversation.
- **Repeat `instructions`** on later turns when role/style consistency matters.
- **Log response IDs** in apps for debugging and traceability.
- **Reset conversation state** when switching topics completely.
- **Use manual history** only when you want explicit control over every prior turn.
- **Inspect usage** if you are monitoring token consumption.


## Suggested exercises

1. Modify the helper so it stores timestamps for each turn.  
2. Add a method that exports the chat history to JSON.  
3. Create two separate `StatefulResponsesChat` objects and compare topic isolation.  
4. Add `temperature` and `max_output_tokens` controls to the helper.  
5. Extend the notebook with streaming examples.


## Summary

This notebook demonstrated how to maintain conversation state with the Responses API using `previous_response_id`.

The main idea is:

1. make an initial `responses.create(...)` call  
2. save the returned `response.id`  
3. pass it into the next call as `previous_response_id`  
4. repeat `instructions` whenever you need them to remain active

That pattern is one of the cleanest ways to build stateful multi-turn interactions with the OpenAI Responses API.
